In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import umap
import numpy as np

from histomining.models.foundation_models import load_model

from histopathossl.training.dataset import TileDataset
from histopathossl.utils import get_device
from histopathossl.models.moco_ligthing import MoCoV2Lightning
from histopathossl.data.tcga_ut_utils import (
    get_train_val_test_tile_paths,
    get_eval_transformations,
)

In [ ]:
train_tile_paths, val_tile_paths, test_tile_paths = get_train_val_test_tile_paths()

In [ ]:
label_map = {
    "Lung_adenocarcinoma": 0,
    "Lung_squamous_cell_carcinoma": 1,
}

In [ ]:
labels_val = [f.parents[2].name for f in val_tile_paths]
# labels_val = [label_map[label] for label in labels_val]

In [ ]:
device = get_device(gpu_id=0)

In [ ]:
model = MoCoV2Lightning.load_from_checkpoint(
    "/home/valentin/workspaces/histopathossl/models/mocov2-tcga-ut.ckpt", map_location=device,
)
backbone = model.encoder_q
backbone.fc = torch.nn.Identity()
backbone.eval()

In [ ]:
transform = get_eval_transformations()
val_loader = DataLoader(
    TileDataset(
        val_tile_paths,
        transform=transform,
    ),
    batch_size=32,
    num_workers=8,
    shuffle=False,
)

In [ ]:
embeddings = []
with torch.no_grad():
    for batch in tqdm(val_loader):
        batch = batch.to(device)
        emb = backbone(batch)
        embeddings.append(emb.cpu().numpy())

# Combine all batches of embeddings
embeddings = np.vstack(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
numeric_labels = np.array([label_map[label] for label in labels_val])

In [ ]:
# Generate UMAP projection
reducer = umap.UMAP(random_state=41)
embedding = reducer.fit_transform(embeddings)

In [ ]:
unique_labels = list(set(labels_val))

In [ ]:
# Plot with proper labels
plt.figure(figsize=(12, 10))
scatter = plt.scatter(
    embedding[:, 0], 
    embedding[:, 1], 
    c=numeric_labels, 
    cmap='tab20', 
    alpha=0.7, 
    s=5
)

# Add legend
legend1 = plt.legend(
    scatter.legend_elements()[0], 
    unique_labels,
    title="Cancer Types",
    loc="upper right"
)
plt.gca().add_artist(legend1)

plt.title('UMAP projection of validation tiles', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
foundation_model, preprocess, embedding_dim, autocast_dtype = load_model("H-optimus-0", device=device)

In [ ]:
val_loader = DataLoader(
    TileDataset(
        val_tile_paths,
        transform=transforms.Compose(
            [
                transforms.CenterCrop(224),
                preprocess,
            ]
        ),
    ),
    batch_size=32,
    num_workers=8,
    shuffle=False,
)

In [ ]:
embeddings = []
with torch.inference_mode():
    with torch.autocast(device_type=device.type, dtype=autocast_dtype):
        for batch in tqdm(val_loader):
            batch = batch.to(device)
            emb = foundation_model(batch)
            embeddings.append(emb.cpu().numpy())

# Combine all batches of embeddings
embeddings = np.vstack(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Generate UMAP projection
reducer = umap.UMAP(random_state=41)
embedding = reducer.fit_transform(embeddings)

In [ ]:
# Plot with proper labels
plt.figure(figsize=(12, 10))
scatter = plt.scatter(
    embedding[:, 0], 
    embedding[:, 1], 
    c=numeric_labels, 
    cmap='tab20', 
    alpha=0.7, 
    s=5
)

# Add legend
legend1 = plt.legend(
    scatter.legend_elements()[0], 
    unique_labels,
    title="Cancer Types",
    loc="upper right"
)
plt.gca().add_artist(legend1)

plt.title('UMAP projection of validation tiles', fontsize=16)
plt.tight_layout()
plt.show()